In [1]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated, List, Dict, Any

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

# --- CARICAMENTO DATI DAI FILE JSON ALLEGATI ---
DATA_DIR = Path("data")

with open(DATA_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS_DB = {c["claim_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "customers.json", "r", encoding="utf-8") as f:
    CUSTOMERS_DB = {c["customer_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "policies.json", "r", encoding="utf-8") as f:
    POLICIES_DB = {p["policy_id"]: p for p in json.load(f)} #

with open(DATA_DIR / "requirements.json", "r", encoding="utf-8") as f:
    REQUIREMENTS_DB = {r["loss_type"]: r for r in json.load(f)} #

llm = ChatOllama(model="llama3.2", temperature=0)


# ==============================================================================
# 2. DEFINIZIONE DEI TOOL PER LE 6 TASK
# ==============================================================================
@tool
def verifica_copertura_e_tipo_polizza(policy_id: str, loss_type: str) -> str:
    """Task 3 e Task 4: Verifica se la polizza è attiva, se copre il tipo di evento (Task 3)

    e restituisce il tipo di assicurazione e prodotto (Task 4).
    """
    policy = POLICIES_DB.get(policy_id) #
    if not policy:
        return f"RIFIUTATO — Polizza '{policy_id}' non trovata a sistema." #
    
    if not policy.get("active", False): #
        return f"RIFIUTATO — La polizza '{policy_id}' risulta DISATTIVA." #
        
    coverage = policy.get("coverage", "") #
    if coverage != loss_type: #
        return f"RIFIUTATO — Il tipo evento '{loss_type}' non è coperto dal tipo garanzia '{coverage}'." #
        
    prodotto = policy.get("product", "Sconosciuto") #
    return f"OK — Polizza attiva. Prodotto: '{prodotto}', Copertura: '{coverage}'." #


@tool
def analizza_documenti(claim_documents: List[str], loss_type: str) -> str:
    """Task 1 e Task 6: Ispeziona i documenti forniti dal cliente (Task 6)

    e verifica se tutti i documenti richiesti sono presenti (Task 1).
    """
    req = REQUIREMENTS_DB.get(loss_type) #
    if not req:
        return f"ERRORE — Tipo evento '{loss_type}' non presente nei requisiti." #
        
    req_docs = set(req.get("required_documents", [])) #
    provided_docs = set(claim_documents) #
    
    mancanti = list(req_docs - provided_docs) #
    
    if mancanti:
        return f"INCOMPLETO — Documenti forniti: {list(provided_docs)}. Documenti mancanti: {mancanti}." #
    
    return f"OK — Documentazione completa. Documenti forniti: {list(provided_docs)}." #


@tool
def calcola_budget_e_limiti(estimated_damage: float, policy_id: str, loss_type: str) -> str:
    """Task 2 e Task 5: Verifica franchigia (tetto minimo) e massimale (tetto massimo) (Task 5),

    e controlla il rientro nel budget di approvazione automatica (Task 2).
    """
    policy = POLICIES_DB.get(policy_id) #
    req = REQUIREMENTS_DB.get(loss_type) #
    
    if not policy or not req:
        return "ERRORE — Dati polizza o requisiti mancanti." #
        
    deductible = policy.get("deductible", 0) #
    max_claim = policy.get("max_claim", 0) #
    auto_max = req.get("automatic_review_max", 0) #
    
    # Check 1: Sotto la franchigia (tetto minimo)
    if estimated_damage <= deductible: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) inferiore o pari alla franchigia/tetto minimo ({deductible}€)." #
        
    # Check 2: Supera il massimale (tetto massimo)
    if estimated_damage > max_claim: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) supera il massimale/tetto massimo della polizza ({max_claim}€)." #
        
    # Check 3: Rientro nel budget per approvazione automatica
    if estimated_damage > auto_max: #
        return f"ESCALATION — Danno stimato ({estimated_damage}€) supera la soglia di approvazione automatica ({auto_max}€). Richiesta revisione manuale." #
        
    return f"OK — Danno stimato ({estimated_damage}€) compreso tra franchigia ({deductible}€) e massimale ({max_claim}€), ed entro la soglia automatica ({auto_max}€)." #



In [2]:
# TEST DI VERIFICA TRASPARENZA E CORRETTEZZA DATI ESTRATTI

def verifica_correttezza_tool(claim_id_da_testare: str):
    print(f"\n============================================================")
    print(f" 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: {claim_id_da_testare}")
    print(f"============================================================")
    
    # 1. Recuperiamo il sinistro dal DB
    claim = CLAIMS_DB.get(claim_id_da_testare)
    if not claim:
        print(f"Errore: Il sinistro {claim_id_da_testare} non esiste nei dati!")
        return

    # Estrazione campi dal sinistro
    policy_id = claim["policy_id"]
    loss_type = claim["loss_type"]
    estimated_damage = claim["estimated_damage"]
    claim_docs = claim["documents"]

    # Recuperiamo i dati corrispondenti dagli altri file JSON
    policy = POLICIES_DB.get(policy_id, {})
    requirement = REQUIREMENTS_DB.get(loss_type, {})

    # STAMPA DEI DATI REALI CONTENUTI NEI JSON 
    print("\n 1. DATI GREZZI PRESENTI NEI FILE JSON:")
    print(f"   • Sinistro ({claim_id_da_testare}): Danno={estimated_damage}€, Documenti Inviati={claim_docs}")
    print(f"   • Polizza ({policy_id}): Attiva={policy.get('active')}, Prodotto={policy.get('product')}, Copertura={policy.get('coverage')}")
    print(f"   • Regole Polizza: Franchigia={policy.get('deductible')}€, Massimale={policy.get('max_claim')}€")
    print(f"   • Requisiti ({loss_type}): Documenti Richiesti={requirement.get('required_documents')}, Max Automatico={requirement.get('automatic_review_max')}€")

    # STAMPA DELLE RISPOSTE GENERATE DAI TOOL
    print("\n 2. RISPOSTE RESTITUITE DAI TUOI TOOL:")

    # Test Tool 1: Copertura e Tipo Polizza
    res_copertura = verifica_copertura_e_tipo_polizza.invoke({
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Copertura:   '{res_copertura}'")

    # Test Tool 2: Documenti
    res_docs = analizza_documenti.invoke({
        "claim_documents": claim_docs, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Documenti:   '{res_docs}'")

    # Test Tool 3: Budget e Limiti
    res_budget = calcola_budget_e_limiti.invoke({
        "estimated_damage": estimated_damage, 
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Budget:      '{res_budget}'")

# --- ESEGUIAMO IL TEST SUI DIVERSI SINISTRI DEL VOSTRO PROGETTO ---
# Test 1: Sinistro CL001 (Caso perfetto)
verifica_correttezza_tool("CL001")

# Test 2: Sinistro CL002 (Caso con documenti mancanti e importo alto)
verifica_correttezza_tool("CL002")

# Test 3: Sinistro CL003 (Caso polizza disattiva)
verifica_correttezza_tool("CL003")




 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL001

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL001): Danno=1200€, Documenti Inviati=['photo', 'police_report']
   • Polizza (POL001): Attiva=True, Prodotto=Auto, Copertura=collision
   • Regole Polizza: Franchigia=250€, Massimale=15000€
   • Requisiti (collision): Documenti Richiesti=['photo', 'police_report'], Max Automatico=3000€

 2. RISPOSTE RESTITUITE DAI TUOI TOOL:
   ► Tool Copertura:   'OK — Polizza attiva. Prodotto: 'Auto', Copertura: 'collision'.'
   ► Tool Documenti:   'OK — Documentazione completa. Documenti forniti: ['police_report', 'photo'].'
   ► Tool Budget:      'OK — Danno stimato (1200.0€) compreso tra franchigia (250€) e massimale (15000€), ed entro la soglia automatica (3000€).'

 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL002

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL002): Danno=7000€, Documenti Inviati=['photo']
   • Polizza (POL002): Attiva=True, Prodotto=Home, Copertu

In [3]:
# ==============================================================================
# 2. DEFINIZIONE DELLO STATO CONDIVISO (StatoRichiesta)
# ==============================================================================
class StatoRichiesta(TypedDict, total=False):
    # Input
    richiesta:              str     # messaggio originale dell'utente
    claim_id:               str     # identificativo pratica
    customer_policy_id:     str     # ID polizza dichiarata
    customer_loss_type:     str     # evento dichiarato (collision, water_damage, theft)
    estimated_damage:       int     # stima economica dell'indennizzo
    documents:              list    # documenti allegati dal cliente

    # Dati arricchiti da database
    customer_name:          str     # nome intestatario
    risk_tier:              str     # fascia di rischio cliente
    policy_id:              str     # codice polizza a sistema
    product:                str     # linea prodotto (Auto / Home)
    coverage:               str     # garanzia operante
    deductible:             int     # franchigia
    max_claims:             int     # massimale
    active:                 bool    # stato validita polizza

    # Parametri specifici della garanzia
    loss_type:              str     # tipologia sinistro confermata
    required_documents:     list    # documentazione obbligatoria
    automatic_review_max:   int     # limite per gestione automatizzata

    # Stato del processo decisionale
    esito:                  str     # APPROVATO | ESCALATION | RIFIUTO | AMBIGUO
    motivazione:            str     # dettaglio sintetico dell'esito
    risposta_finale:        str     # comunicazione per il cliente/operatore
    retry_count:            int     # contatore tentativi di chiarimento
    chiarimento:            str     # richiesta inviata in caso di documentazione incompleta
    errori:                 list    # incongruenze riscontrate


# ==============================================================================
# 3. FUNZIONI DI ELABORAZIONE DEL FLUSSO
# ==============================================================================

def recupera_dati_pratica(state: StatoRichiesta) -> StatoRichiesta:
    """Interroga i database per associare al sinistro le informazioni

    di polizza, cliente e requisiti di garanzia.
    """
    print("Recupero ed estrazione dati della pratica...")
    
    claim_id = state.get("claim_id", "")
    claim_info = CLAIMS_DB.get(claim_id, {})
    
    # Recupero attributi con fallback sui valori forniti nell'input
    policy_id = state.get("customer_policy_id") or claim_info.get("policy_id", "")
    loss_type = state.get("customer_loss_type") or claim_info.get("loss_type", "")
    damage = state.get("estimated_damage") if state.get("estimated_damage") is not None else claim_info.get("estimated_damage", 0)
    docs = state.get("documents") if state.get("documents") is not None else claim_info.get("documents", [])
    
    # Lettura record nei rispettivi DB
    policy_info = POLICIES_DB.get(policy_id, {})
    customer_info = CUSTOMERS_DB.get(policy_info.get("customer_id", ""), {})
    req_info = REQUIREMENTS_DB.get(loss_type, {})

    return {
        "claim_id": claim_id,
        "customer_policy_id": policy_id,
        "customer_loss_type": loss_type,
        "estimated_damage": damage,
        "documents": docs,
        
        "customer_name": customer_info.get("name", "Cliente"),
        "risk_tier": customer_info.get("risk_tier", "standard"),
        "policy_id": policy_id,
        "product": policy_info.get("product", "N/A"),
        "coverage": policy_info.get("coverage", "N/A"),
        "deductible": policy_info.get("deductible", 0),
        "max_claims": policy_info.get("max_claim", 0),
        "active": policy_info.get("active", False),
        
        "loss_type": loss_type,
        "required_documents": req_info.get("required_documents", []),
        "automatic_review_max": req_info.get("automatic_review_max", 0),
        
        "retry_count": state.get("retry_count", 0),
        "errori": []
    }


def valuta_richiesta_sinistro(state: StatoRichiesta) -> StatoRichiesta:
    """Verifica la validita della polizza, la copertura dell'evento,

    la completezza documentale e i limiti economici stabiliti.
    """
    print("Analisi e verifica delle regole contrattuali...")
    
    errori = []
    
    # 1. Verifica esistenza e stato di validita della polizza
    if not state.get("policy_id") or state.get("product") == "N/A":
        errori.append(f"Polizza '{state.get('customer_policy_id')}' non trovata a sistema.")
        return {
            "esito": "RIFIUTO",
            "motivazione": "Polizza inesistente nei registri aziendali.",
            "errori": errori
        }

    if not state.get("active", False):
        errori.append(f"La polizza {state.get('policy_id')} risulta non attiva.")
        return {
            "esito": "RIFIUTO",
            "motivazione": f"La polizza {state.get('policy_id')} è risultata disattivata.",
            "errori": errori
        }

    # 2. Controllo coerenza tra garanzia acquistata ed evento dichiarato
    if state.get("coverage") != state.get("customer_loss_type"):
        errori.append(
            f"Evento '{state.get('customer_loss_type')}' non coperto dalla garanzia '{state.get('coverage')}'."
        )
        return {
            "esito": "RIFIUTO",
            "motivazione": f"Tipologia di danno non coperta dalla polizza {state.get('policy_id')}.",
            "errori": errori
        }

    # 3. Controllo allegati obbligatori
    docs_inviati = set(state.get("documents", []))
    docs_richiesti = set(state.get("required_documents", []))
    mancanti = list(docs_richiesti - docs_inviati)

    if mancanti:
        errori.append(f"Documenti mancanti: {mancanti}.")
        if state.get("retry_count", 0) < 2:
            return {
                "esito": "AMBIGUO",
                "motivazione": f"Documentazione incompleta. Mancano: {mancanti}.",
                "errori": errori
            }
        else:
            return {
                "esito": "RIFIUTO",
                "motivazione": f"Richiesta respinta per mancata integrazione dei documenti ({mancanti}).",
                "errori": errori
            }

    # 4. Controllo franchigia, massimale e soglia automatica
    danno = state.get("estimated_damage", 0)
    franchigia = state.get("deductible", 0)
    massimale = state.get("max_claims", 0)
    soglia_automatica = state.get("automatic_review_max", 0)

    if danno <= franchigia:
        errori.append(f"Importo stimato ({danno}€) inferiore o pari alla franchigia ({franchigia}€).")
        return {
            "esito": "RIFIUTO",
            "motivazione": f"Danno stimato ({danno}€) entro la franchigia contrattuale di {franchigia}€.",
            "errori": errori
        }

    if danno > massimale:
        errori.append(f"Importo stimato ({danno}€) superiore al massimale di polizza ({massimale}€).")
        return {
            "esito": "RIFIUTO",
            "motivazione": f"Danno stimato ({danno}€) eccede il massimale coperto ({massimale}€).",
            "errori": errori
        }

    if danno > soglia_automatica:
        return {
            "esito": "ESCALATION",
            "motivazione": f"Importo ({danno}€) superiore al limite per liquidazione automatica ({soglia_automatica}€). Pratica trasferita a un perito.",
            "errori": []
        }

    return {
        "esito": "APPROVATO",
        "motivazione": "Requisiti contrattuali e documentali pienamente soddisfatti.",
        "errori": []
    }


def elabora_risposta_cliente(state: StatoRichiesta) -> StatoRichiesta:
    """Genera la comunicazione formale di riscontro per il cliente o l'operatore."""
    print("Composizione della comunicazione finale...")
    
    esito = state.get("esito", "RIFIUTO")
    cliente = state.get("customer_name", "Cliente")
    claim_id = state.get("claim_id", "N/A")
    motivazione = state.get("motivazione", "")
    
    prompt = f"""
    Sei l'Assistente AI per la gestione delle Pratiche Assicurative.
    Redigi un riscontro formale e chiaro per il cliente sulla seguente pratica:

    - Nome Cliente: {cliente}
    - Codice Pratica: {claim_id}
    - Prodotto / Polizza: {state.get('product')} ({state.get('policy_id')})
    - Tipo Sinistro: {state.get('customer_loss_type')}
    - Importo Richiesto: {state.get('estimated_damage')}€
    - Franchigia: {state.get('deductible')}€ | Massimale: {state.get('max_claims')}€
    - ESITO VALUTAZIONE: {esito}
    - MOTIVAZIONE: {motivazione}

    Istruzioni:
    - Se l'esito è APPROVATO: Conferma l'accoglimento della richiesta spiegando la liquidazione.
    - Se l'esito è ESCALATION: Spiega che la pratica è in corso di valutazione da parte del reparto peritale.
    - Se l'esito è RIFIUTO: Spiega in modo cortese le ragioni del diniego.
    - Se l'esito è AMBIGUO: Richiedi l'invio della documentazione mancante.
    """
    
    risposta = llm.invoke([HumanMessage(content=prompt)]).content
    return {"risposta_finale": risposta}


def richiedi_integrazione_documenti(state: StatoRichiesta) -> StatoRichiesta:
    """Predispone la richiesta di integrazione documentale per il cliente."""
    print("Gestione richiesta integrazione allegati...")
    
    retry = state.get("retry_count", 0) + 1
    mancanti = state.get("errori", [])
    
    messaggio = (
        f"Integrazione richiesta (#{retry}) per la pratica {state.get('claim_id')}: "
        f"per procedere è necessario inviare i seguenti documenti: {mancanti}"
    )
    
    return {
        "retry_count": retry,
        "chiarimento": messaggio,
        "risposta_finale": messaggio
    }


# ==============================================================================
# 4. ROUTING E STRUTTURA DEL FLUSSO
# ==============================================================================

def determina_flusso_pratica(state: StatoRichiesta) -> str:
    """Indirizza l'esecuzione in base all'esito della valutazione."""
    if state.get("esito") == "AMBIGUO":
        return "richiedi_integrazione"
    return "elabora_risposta"


# Assemblaggio del grafo
builder = StateGraph(StatoRichiesta)

# Registrazione passaggi
builder.add_node("recupera_dati", recupera_dati_pratica)
builder.add_node("valuta_richiesta", valuta_richiesta_sinistro)
builder.add_node("elabora_risposta", elabora_risposta_cliente)
builder.add_node("richiedi_integrazione", richiedi_integrazione_documenti)

# Definizione sequenza
builder.add_edge(START, "recupera_dati")
builder.add_edge("recupera_dati", "valuta_richiesta")

builder.add_conditional_edges(
    "valuta_richiesta",
    determina_flusso_pratica,
    {
        "richiedi_integrazione": "richiedi_integrazione",
        "elabora_risposta": "elabora_risposta"
    }
)

builder.add_edge("elabora_risposta", END)
builder.add_edge("richiedi_integrazione", END)

# Compilazione workflow
agente_valutazione = builder.compile()
print("Flusso di valutazione compilato con successo!")


# ==============================================================================
# 5. TEST PRATICO
# ==============================================================================
if __name__ == "__main__":
    for cid in ["CL001", "CL002", "CL003", "CL004"]:
        print(f"\n============================================================")
        print(f" 🧪 ESECUZIONE VALUTAZIONE: {cid}")
        print(f"============================================================")
        
        risultato = agente_valutazione.invoke({
            "claim_id": cid,
            "richiesta": f"Elaborazione pratica {cid}"
        })
        
        print(f"\n📌 ESITO     : {risultato.get('esito')}")
        print(f"💡 MOTIVAZIONE: {risultato.get('motivazione')}")
        print(f"💬 RISPOSTA  :\n{risultato.get('risposta_finale')}")

Flusso di valutazione compilato con successo!

 🧪 ESECUZIONE VALUTAZIONE: CL001
Recupero ed estrazione dati della pratica...
Analisi e verifica delle regole contrattuali...
Composizione della comunicazione finale...

📌 ESITO     : APPROVATO
💡 MOTIVAZIONE: Requisiti contrattuali e documentali pienamente soddisfatti.
💬 RISPOSTA  :
Gentile Pietro Sala,

Siamo lieti di informarLa che la sua richiesta di indennizzo per il sinistro della sua auto, codice pratica CL001, è stata valutata e approvata.

La liquidazione dell'indennizzo sarà effettuata secondo i requisiti contrattuali e documentali pienamente soddisfatti, come richiesto. L'importo richiesto di 1200€ sarà correttamente calcolato e versato al cliente.

La motivazione dell'approvazione è la seguente: i requisiti contrattuali e documentali presentati sono stati pienamente soddisfatti, garantendo una valutazione equa e giusta della richiesta.

La liquidazione dell'indennizzo sarà effettuata entro i prossimi 5 giorni lavorativi, una vol